In [ ]:
#
# This is a single, complete script for the freeCodeCamp SMS Text Classifier.
# You can paste this entire block into one cell in Google Colab and run it.
#

# =================================================================================
# CELL 1: SETUP AND IMPORTS
# =================================================================================
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
# CORRECTED IMPORT PATH FOR TextVectorization
from tensorflow.keras.layers import TextVectorization
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)


# =================================================================================
# CELL 2: GET DATA AND PREPARE LABELS/FEATURES
# =================================================================================
# get data files
!wget -q https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget -q https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

# Load the datasets into pandas DataFrames
train_df = pd.read_csv(train_file_path, sep="\t", header=None, names=['label', 'message'])
test_df = pd.read_csv(test_file_path, sep="\t", header=None, names=['label', 'message'])

# Convert labels to numerical format (ham=0, spam=1)
train_df['label'] = train_df['label'].map({'ham': 0, 'spam': 1})
test_df['label'] = test_df['label'].map({'ham': 0, 'spam': 1})

# Extract labels and messages
train_labels = train_df['label'].values
train_messages = train_df['message'].values

test_labels = test_df['label'].values
test_messages = test_df['message'].values


# =================================================================================
# CELL 3: TEXT VECTORIZATION (TOKENIZATION & PADDING)
# =================================================================================
# Define vocabulary size and sequence length
VOCAB_SIZE = 1000
MAX_SEQUENCE_LENGTH = 100

# Create a TextVectorization layer
# This layer will handle tokenizing, indexing, and padding the text.
vectorize_layer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH)

# Adapt the layer to our training text to build the vocabulary
vectorize_layer.adapt(train_messages)


# =================================================================================
# CELL 4: BUILD AND TRAIN THE MODEL
# =================================================================================
# Build the model
model = tf.keras.Sequential([
    vectorize_layer,
    keras.layers.Embedding(
        input_dim=len(vectorize_layer.get_vocabulary()),
        output_dim=64,
        mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid') # Sigmoid for binary classification
])

# Compile the model
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    optimizer='adam',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    train_messages,
    train_labels,
    epochs=10,
    validation_data=(test_messages, test_labels),
    validation_steps=30,
    verbose=0 # Changed to 0 to reduce output spam on re-runs
)


# =================================================================================
# CELL 5: PREDICTION FUNCTION
# =================================================================================
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
    # Convert the input text into a NumPy array, which the model expects.
    input_array = np.array([pred_text])
    prediction_prob = model.predict(input_array)[0][0]

    if prediction_prob > 0.5:
        label = "spam"
    else:
        label = "ham"

    prediction = [prediction_prob, label]
    return prediction

# Example usage
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)


# =================================================================================
# CELL 6: FINAL TEST (PROVIDED BY FREECODECAMP)
# =================================================================================
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
